# 9차시 실습 — 머신러닝 예측 모델 만들고 새 로트 예측하기

## 1단계. 데이터 읽고 빈칸 정리하기

공정 기록 750행(교육용 가상 데이터)입니다. `합격여부`: 1 = 합격, -1 = 불합격. 설비 4대(EQ-01~EQ-04)가 여러 공정 기록을 함께 남겼습니다(교육용 가상 기록). 빈칸(NaN)이 있으면 모델 종류에 따라 오류가 나거나 결과가 달라지므로, 빈칸 행을 먼저 지웁니다.

- 빈칸 행 지우기는 5차시에 배운 함수입니다. 결과가 `(735, 9)`이면 맞습니다.

In [ ]:
import pandas as pd

df = pd.read_csv("week09_pass_fail_train.csv", encoding="utf-8")
print(df.shape)

In [ ]:
print(df.isna().sum())

In [ ]:
# TODO: 빈칸이 있는 행을 지우는 함수 이름을 채우세요. (결과: (735, 9))
df = df.____()
print(df.shape)

## 2단계. Feature와 Label 나누고, 학습용·시험용으로 나누기

X(Feature) = 모델이 보고 판단할 열 6개, y(Label) = 맞혀야 할 정답 열 `합격여부`. 기록의 80%(588행)로 학습하고, 떼어 둔 20%(147행)로 채점합니다.

**주의**: X에 `합격여부`를 넣으면 정답을 보고 맞히는 셈이라 정확도가 100%에 가깝게 나옵니다(정답 누설). `공정명`·`설비번호`는 글자 열이라 오늘은 넣지 않습니다.

In [ ]:
features = ["온도_C", "압력_Pa", "가스유량_slm", "두께_nm", "진공도_mTorr", "습도_pct"]
X = df[features]
y = df["합격여부"]
print(X.shape, y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), len(X_test))

> 찾아보기: "결정 트리 스무고개" — 강의 자료의 ChatGPT 질문을 써 보세요.

## 3단계. 결정 트리 학습하고 채점하기

오늘의 미션: 이 모델이 "무조건 합격"이라고 찍는 것(기준선)보다 나은지 비교합니다.

- 비교 기준: 시험용 147개를 모두 "합격"이라고 찍으면 정확도는 78.9%입니다(기준선).

**주의**: 학습하는 셀을 실행하기 전에 정확도 셀을 실행하면 `NotFittedError`가 납니다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=4, random_state=42)
# TODO: 학습용 X와 y로 모델을 학습시키는 함수 이름을 채우세요.
model.____(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, model.predict(X_test))
print(f"정확도: {round(acc * 100, 1)}%")

이 모델은 시험용 불합격 31개 중 9개만 잡고 22개를 합격으로 놓쳤습니다. '무조건 합격'은 31개를 모두 놓칩니다. 정확도가 비슷해도 하는 일은 다릅니다 — 실습지 문제 4에서 Orange 헷갈림 표로 같은 것을 셉니다.

> 찾아보기: "정확도의 함정 불균형 데이터" — 강의 자료의 ChatGPT 질문을 써 보세요.

## 4단계. 새 로트 예측하기

새 로트 파일(교육용 가상 데이터)에는 정답(`합격여부`) 열이 없습니다. 모델이 맞혀야 할 대상입니다. `예측` 열도 `합격여부`와 같은 규칙입니다: 1 = 합격, -1 = 불합격.

- 예측하는 함수는 3단계 정확도 줄의 `model.predict(X_test)`에서 이미 보았습니다. 학습 때와 **같은 열 6개**(`features`)를 넣습니다.

**주의**: `new_lots` 전체를 넣으면 `ValueError: The feature names should match those that were passed during fit.`로 시작하는 오류가 납니다. `new_lots["합격여부"]`를 찾으면 `KeyError: '합격여부'`가 납니다 — 새 로트에는 정답 열이 없습니다.

In [ ]:
new_lots = pd.read_csv("week09_new_lots_to_predict.csv", encoding="utf-8")
print(new_lots.shape)

In [ ]:
# TODO: 학습한 모델로 새 로트의 합격/불합격을 예측하는 함수 이름을 채우세요.
new_lots["예측"] = model.____(new_lots[features])
print(new_lots[["공정명", "설비번호", "온도_C", "예측"]])

In [ ]:
print(f"새 로트 {len(new_lots)}개 중 불합격 예측 {(new_lots['예측'] == -1).sum()}개")

> 찾아보기: "예측 확률의 뜻" — 강의 자료의 ChatGPT 질문을 써 보세요.

## 5단계. ChatGPT 체험 — 보고서의 숫자 확인하기

ChatGPT에 아래 프롬프트를 그대로 붙여 넣으세요(강의 자료와 같은 글자입니다).

> 아래는 교육용 가상 데이터로 만든 반도체 공정 합격/불합격 예측 결과야. 팀장에게 보낼 3문장 보고서를 써 줘. 결과를 원인으로 단정하지 말고 "원인 후보"라는 표현을 쓰고, 마지막에 가상 데이터라는 점을 밝혀 줘.
> - 결정 트리 정확도(시험용 147개): 80.3%
> - 기준선(모두 합격이라고 찍었을 때의 정확도): 78.9%
> - Orange Confusion Matrix(결정 트리, 10겹 교차검증, 750건): 실제 불합격 155건 중 119건을 합격으로 예측(놓침)
> - 새 로트 9개 예측: 불합격 1개(3번째 로트), 합격 8개

**응답 예시 — 실제 AI 응답 아님 · 틀린 곳이 숨어 있음**(ChatGPT를 쓸 수 없으면 이 예시로 틀린 곳을 찾으세요)

> 새 로트 9개 중 결정 트리가 불합격으로 예측한 로트는 2개이며, 3번째 로트(식각·EQ-04, 온도 289.8도)가 그중 하나입니다. 모델 정확도는 80.3%로 모두 합격이라고 찍는 기준선 78.9%와 큰 차이가 없고, Orange 교차검증에서는 실제 불합격 155건 중 119건을 합격으로 놓쳤습니다. 3번째 로트는 온도가 낮아서 불합격이 생깁니다(이 결과는 교육용 가상 데이터로 만든 것입니다).

**숫자 대조** — 응답 예시나 내가 받은 보고서의 숫자를 아래 셀의 출력과 하나씩 맞춰 보세요. 숫자가 틀린 곳과 "원인"이라고 단정한 곳을 찾아 실습지 문제 5에서 고쳐 씁니다.

In [ ]:
print(f"정확도: {round(acc * 100, 1)}%")
print(f"기준선(모두 합격): {round((y_test == 1).mean() * 100, 1)}%")
print(f"불합격 예측: {(new_lots['예측'] == -1).sum()}개")
print(new_lots[new_lots["예측"] == -1][["공정명", "설비번호", "온도_C"]])

## 6단계. 반도체 데이터 실습

실습지 문제 1~2와 같은 문제입니다. Orange Tree Viewer의 첫 갈림길은 `온도_C`였습니다. 그렇다면 `온도_C` 한 열만 보는 모델은 6개 열 모델(3단계)이나 기준선(78.9%)과 비교해 어떨까요?

### 문제 1 — `온도_C` 한 열로 Feature 만들고, 학습용·시험용으로 나누기

- 2단계 `features`처럼 열 이름을 리스트(대괄호)에 넣습니다. 열이 하나여도 리스트로 씁니다.
- 나누는 줄은 2단계와 같고, 이름에만 `1`을 붙였습니다. y는 2단계의 `y`를 그대로 씁니다.

**주의**: 열 이름은 따옴표로 감쌉니다. `X1 = df[features]`로 쓰면 오류 없이 6개 열이 들어가 3단계와 같은 모델이 됩니다. `(735,)`가 나오면 대괄호가 빠진 것입니다(`temp_features = "온도_C"`). 그대로 학습하면 `ValueError: Expected a 2-dimensional container`로 시작하는 오류가 납니다.

In [ ]:
# TODO: 온도 열 이름 하나를 리스트에 넣고, 그 리스트 변수 이름으로 X1을 만드세요.
temp_features = [____]
X1 = df[____]
print(X1.shape)

In [ ]:
# TODO: 2단계와 같은 모양입니다. 정답 y의 변수 이름을 두 곳에 채우세요. (결과: 588 147)
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, ____, test_size=0.2, random_state=42, stratify=____)
print(len(X1_train), len(X1_test))

### 문제 2 — 결정 트리 학습하고 정확도 구하기 (직접 쓰기)

- 빈칸이 없습니다. 3단계의 학습·정확도 코드와 같은 모양으로 직접 씁니다.
- 모델 이름은 `model1`, 정확도 이름은 `acc1`, 데이터는 문제 1의 `X1_train`·`y1_train`·`X1_test`·`y1_test`로 바꿉니다. 설정(`max_depth=4, random_state=42`)은 그대로 둡니다.
- 출력 문장은 `온도_C 한 열 정확도: …%`로 씁니다. 3단계 정확도(6개 열), 기준선 78.9%와 비교하세요.

**주의**: `X_test`(6개 열)와 `X1_test`(1개 열)를 섞어 쓰면 `ValueError`가 납니다. `fit`을 실행하기 전에 정확도 줄을 실행하면 `NotFittedError`가 납니다. 82.7%가 나오면 학습용(`X1_train`, `y1_train`)으로 채점한 것입니다. 이미 본 문제라 점수가 부풀려집니다. 채점은 시험용(`X1_test`, `y1_test`)으로 합니다. 80.3%가 나오면 문제 1의 `X1`에 6개 열이 들어갔거나, 정확도 줄에 `model`·`X_test`가 그대로 남은 것입니다.

In [ ]:
# TODO: 여기에 직접 써 보세요 — model1 만들기 → fit(X1_train, y1_train) → acc1 구하기 → print